# Python-oop-Encapsulation & Property Decorators

Welcome to Part 6 of my Python OOP series! 
* Previous Notebook: Part 5 - Magic Methods (https://www.kaggle.com/code/aymanhatabah/python-oop-magic-method-dunder-method)

### The Core Concept: Encapsulation
In professional programming, we don't want external scripts to change our object's internal data directly. We use **Encapsulation** to restrict access. In Python, we do this using:
1. **Protected (`_variable`)**: A warning to other developers saying *"This is internal, don't change it outside the class."*
2. **Private (`__variable`)**: Python physically locks this variable using **Name Mangling** so it cannot be accessed directly from outside.

### Enter Property Decorators
To allow safe access to these hidden variables, we use **Property Decorators** (`@property`). This mimics libraries like **Pandas**, where internal data structures are protected, but exposed safely via attributes like `df.shape` or `df.columns` or ... .

## 1. Protected, Private, and The Getter (@property)
Let's build a `MiniDataFrame`. We will make the raw data grid strictly **Private** (`__data`) because changing it directly can corrupt the dataset. We will make the column names **Protected** (`_columns`).

We will then use a **Getter** (`@property`) to let users read the dataframe's `shape` dynamically without exposing the raw private matrix.

In [1]:
class MiniDataFrame:
    def __init__(self, data: list, columns: list):
        # PRIVATE: Locked inside the vault. External code cannot see or modify __data directly.
        self.__data = data          
        
        # PROTECTED: Internal to this class and its subclasses.
        self._columns = columns    

    @property
    def shape(self) -> tuple:
        """
        The Getter: Calculates the dimensions of the private __data grid.
        It acts like a read-only variable from the outside.
        """
        row_count = len(self.__data)
        column_count = len(self.__data[0]) if row_count > 0 else 0
        return (row_count, column_count)

# --- Testing Encapsulation & Getter ---
my_data = [[23, 8500], [25, 9200], [22, 7100]]
my_cols = ["Age", "Salary"]

df = MiniDataFrame(data=my_data, columns=my_cols)

# 1. Accessing via the safe Getter property
print(f"DataFrame Shape: {df.shape}")  # Output: (3, 2)

# 2. Testing Private Security (This will fail)
try:
    print(df.__data)
except AttributeError:
    print(" SECURITY SUCCESS: Direct access to '__data' is blocked by Python Name Mangling.")

DataFrame Shape: (3, 2)
 SECURITY SUCCESS: Direct access to '__data' is blocked by Python Name Mangling.


## 2. Guarding Data with the Setter (@property.setter)
What if the user wants to rename the columns? If we let them modify `df._columns` directly, they might provide the wrong number of names, causing the system to crash later.

We will create a **Setter** for `columns`. The setter acts like a security guard. It will take the input string, split it (just like Corey Schafer split the full name string), and validate it against the private data width before applying changes.

In [2]:
class SmartDataFrame(MiniDataFrame):
    def __init__(self, data: list, columns: list):
        super().__init__(data, columns)
        # Internal caching system to speed up queries
        self._cache = {"calculated_means": [23.3, 8266.6]}

    @property
    def columns(self) -> list:
        """The Getter: Exposes the protected _columns list safely as read-only by default."""
        return self._columns

    @columns.setter
    def columns(self, column_input: str):
        """
        The Setter: Accepts a comma-separated string, parses it,
        and validates it before updating the protected _columns attribute.
        """
        # Parse the input string into a clean list
        parsed_labels = [label.strip() for label in column_input.split(",")]
        
        # Validation Gate: Does the number of names match our data width?
        required_count = self.shape[1]
        if len(parsed_labels) != required_count:
            raise ValueError(
                f"Validation Error: Data needs {required_count} column names, "
                f"but you provided {len(parsed_labels)}."
            )
            
        # If valid, update the protected attribute and clear old cache
        print("[SYSTEM] Validation passed. Updating column names...")
        self._columns = parsed_labels
        self._cache.clear() 

# --- Testing the Setter ---
df_smart = SmartDataFrame(data=my_data, columns=my_cols)

# Test A: Successful update through the Setter
df_smart.columns = "User_Age , Monthly_Income"
print(f"New Columns: {df_smart.columns}")
print(f"Cache Status: {df_smart._cache}\n")  # Cache is cleared {}

# Test B: Invalid update rejected by the Setter
try:
    df_smart.columns = "InvalidSingleName"
except ValueError as error:
    print(f"SYSTEM BLOCKED ERROR: {error}")

[SYSTEM] Validation passed. Updating column names...
New Columns: ['User_Age', 'Monthly_Income']
Cache Status: {}

SYSTEM BLOCKED ERROR: Validation Error: Data needs 2 column names, but you provided 1.


## 3. Resetting Variables with the Deleter (@property.deleter)
## 3. Extending Properties using Inheritance
When we want to add a **Deleter** to a child class, we must point directly to the parent's property using `@ParentClass.property.deleter`. 

This tells Python to copy the existing Getter and Setter from `SmartDataFrame` and add this new Deleter functionality to it.

In [3]:
class FinalDataFrame(SmartDataFrame):
    """
    Our complete framework combining Private/Protected attributes with Property controls.
    """
    def __init__(self, data: list, columns: list):
        # Pass the arguments correctly to the parent class constructor
        super().__init__(data, columns)

    # FIX: Point directly to the parent class descriptor so Python can find it
    @SmartDataFrame.columns.deleter
    def columns(self):
        """
        The Deleter: Cleans up custom names and falls back to default index strings.
        """
        print("[WARNING] Custom columns deleted. Resetting to default index format...")
        
        # Accessing the getter property to find the data width
        column_width = self.shape[1]
        
        # Generates default index like ['0', '1'] and updates the protected variable
        self._columns = [str(i) for i in range(column_width)]

# --- Testing the Deleter System ---
# Now creating the object works perfectly with arguments
df_final = FinalDataFrame(data=my_data, columns=my_cols)
print(f"Initial columns: {df_final.columns}\n")

# Triggering the deleter using the 'del' keyword
del df_final.columns

print(f"Columns after deletion: {df_final.columns}")

Initial columns: ['Age', 'Salary']

[WARNING] Custom columns deleted. Resetting to default index format...
Columns after deletion: ['0', '1']


## Summary: Encapsulation & Property Reference Table

| OOP Tool | Syntax Example | Visibility / Access | Main Purpose in Frameworks |
| :--- | :--- | :--- | :--- |
| **Protected** | `self._columns` | Class & Subclasses | Warning: Internal variable, use with care. |
| **Private** | `self.__data` | Inside Class Only | Strict lock: Prevents external data corruption. |
| **Getter** | `@property` | Public Read-Only | Computes values on the fly without changing APIs. |
| **Setter** | `@columns.setter` | Public Validated Write | Filters inputs, splits data, and protects integrity. |
| **Deleter** | `@columns.deleter` | Public Cleanup | Securely resets data or wipes out system memory. |